# 🧬 CNN — Identificador de Etnias Culturales (UTKFace)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Problema:** clasificar rostros en 5 grupos étnicos (White, Black, Asian, Indian, Others) a partir de rasgos físicos.
**Modelo:** CNN + Transfer Learning (MobileNetV2). **Dataset:** UTKFace Aligned & Cropped (~20k rostros).

> ⚠️ Uso educativo y responsable: la etnia no se reduce a rasgos físicos; el dataset está sesgado y el modelo **no debe usarse** para discriminar ni vigilar.

**Checklist de entregables cubiertos en este notebook:**
- [x] Selección y descripción del dataset
- [x] Exploración y visualización
- [x] Preprocesamiento y normalización
- [x] División train/val/test
- [x] CNN + aumento de datos + entrenamiento
- [x] Métricas, gráficas, matriz de confusión
- [x] ≥10 predicciones con imágenes
- [x] Guardado del modelo `.h5`


In [ ]:
# --- Entorno Colab: GPU + deps ---
import sys
!nvidia-smi -L 2>/dev/null || echo "Sin GPU visible (usa Entorno > GPU)"
!pip -q install -U pyarrow seaborn scikit-learn pillow pandas
import tensorflow as tf
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Selección y descripción del dataset: UTKFace

| Aspecto | Detalle |
|---|---|
| Fuente | [UTKFace](https://susanqq.github.io/UTKFace/) (Zhang et al., CVPR 2017), solo investigación no comercial |
| Versión | **Aligned & Cropped Faces** (~107 MB, ~20k+ imágenes, un rostro por foto) |
| Etiqueta en el nombre | `[age]_[gender]_[race]_[datetime].jpg` — `race: 0=White, 1=Black, 2=Asian, 3=Indian, 4=Others (Hispanic/Latino/Middle Eastern)` |
| Edad / género | 0–116 años; gender 0=hombre, 1=mujer (estimados con DEX + verificación humana) |
| Variabilidad | Pose, expresión, iluminación, oclusión, resolución |
| Por qué se eligió | Benchmark estándar de etnia, ya recortado a rostro, desbalance real que exige `class_weight` + augmentation |
| Alternativa | FairFace (más balanceado racialmente) |


In [ ]:
# --- Obtención del dataset (fuente funcional verificada) ---
# El enlace original de Drive (carpeta 0BxY...DRUE) está caído (devuelve 404).
# Se usa el espejo de Hugging Face "vtsouval/utkface-cropped": un parquet con
# imagen + edad + género + etnia (23.704 filas, ~111 MB).
from pathlib import Path
import glob, io, random, shutil, urllib.request
DATA_DIR = Path("data/raw/UTKFace")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PARQUET = Path("data/raw/utkface-cropped.parquet")
MUESTRA = Path("data/muestra/UTKFace")  # incluida en el repo (125 imágenes)
CUOTA_POR_ETNIA = 400  # 400 x 5 = 2000 imágenes de trabajo (ajustable)
URL_PARQUET = ("https://huggingface.co/datasets/vtsouval/utkface-cropped/resolve/main/"
               "data/train-00000-of-00001.parquet")

def contar(carpeta):
    return len(glob.glob(str(carpeta / "*.jpg")))

print(f"Imágenes ya presentes: {contar(DATA_DIR)}")
if contar(DATA_DIR) < CUOTA_POR_ETNIA * 5 and contar(MUESTRA) >= 100 and contar(DATA_DIR) == 0:
    # Arranque sin internet: copia la muestra del repo para poder seguir.
    for f in MUESTRA.glob("*.jpg"):
        shutil.copy(f, DATA_DIR / f.name)
    print(f"Muestra local copiada ({contar(DATA_DIR)} imágenes). Sirve para probar sin internet.")

if contar(DATA_DIR) < CUOTA_POR_ETNIA * 5:
    if not PARQUET.exists():
        print("Descargando espejo UTKFace (~111 MB)...")
        req = urllib.request.Request(URL_PARQUET, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(PARQUET, "wb") as f:
            shutil.copyfileobj(r, f)
        print(f"Parquet guardado ({PARQUET.stat().st_size/1e6:.0f} MB).")
    import pyarrow.parquet as pq
    from PIL import Image
    random.seed(42)
    pf = pq.ParquetFile(PARQUET)
    print(f"Extrayendo muestra balanceada ({CUOTA_POR_ETNIA} por etnia, {pf.metadata.num_rows} filas)...")
    por_etnia = {r: [] for r in range(5)}
    for lote in pf.iter_batches(batch_size=2000, columns=["age", "gender", "race"]):
        base = sum(len(v) for v in por_etnia.values())
        for i, (edad, genero, etnia) in enumerate(zip(lote.column("age").to_pylist(),
                                                      lote.column("gender").to_pylist(),
                                                      lote.column("race").to_pylist())):
            if etnia is not None and 0 <= etnia <= 4:
                por_etnia[etnia].append((base + i, int(edad), int(genero)))
    objetivo = {}
    for r in range(5):
        for j, (fila, edad, genero) in enumerate(random.sample(por_etnia[r], CUOTA_POR_ETNIA)):
            objetivo[fila] = (r, j, edad, genero)
    vistas, n = 0, 0
    for lote in pf.iter_batches(batch_size=1000, columns=["image"]):
        for k, img in enumerate(lote.column("image").to_pylist()):
            if vistas + k in objetivo:
                r, j, edad, genero = objetivo[vistas + k]
                datos = img["bytes"] if isinstance(img, dict) else img
                foto = Image.open(io.BytesIO(datos)).convert("RGB")
                foto.save(DATA_DIR / f"{edad}_{genero}_{r}_20240101000000{j:05d}.jpg", "JPEG", quality=90)
                n += 1
        vistas += lote.num_rows
    print(f"Extraídas {n} imágenes.")
print(f"Conjunto listo: {contar(DATA_DIR)} imágenes en {DATA_DIR}")


In [ ]:
# --- Parseo de etiquetas + EDA numerica ---
import re, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter

CLASS_NAMES = ["White", "Black", "Asian", "Indian", "Others"]
CLASES_ES = ["Blanca / Caucásica", "Negra / Afrodescendiente", "Asiática",
             "India / Sudasiática", "Otras (latina, Oriente Medio, mixta…)"]
ES_POR_CLASE = dict(zip(CLASS_NAMES, CLASES_ES))
RE = re.compile(r"^(\d+)_([01])_([0-4])_.*\.jpg$", re.IGNORECASE)

files = sorted(glob.glob("data/raw/UTKFace/*.jpg"))
rows = []
for f in files:
    m = RE.match(Path(f).name)
    if m:
        rows.append((f, int(m.group(1)), int(m.group(2)), int(m.group(3))))
df = pd.DataFrame(rows, columns=["path", "age", "gender", "race"])
df["ethnicity"] = df["race"].map(dict(enumerate(CLASS_NAMES)))
print(f"Total validas: {len(df):,} / archivos: {len(files):,}")
print(df["ethnicity"].value_counts())
print(df["ethnicity"].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="ethnicity", order=CLASS_NAMES, ax=ax[0], palette="viridis")
ax[0].set_title("Distribución por etnia (desbalance real)"); ax[0].set_xticklabels(CLASES_ES, rotation=20, ha="right")
sns.histplot(df["age"], bins=40, kde=False, ax=ax[1]); ax[1].set_title("Edades (0–116)")
sns.countplot(data=df, x="gender", ax=ax[2]); ax[2].set_xticklabels(["Hombre (0)", "Mujer (1)"]); ax[2].set_title("Género")
plt.tight_layout(); plt.show()
df.head(3)

In [ ]:
# --- Exploracion visual: 1 fila por etnia ---
from PIL import Image
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(12, 8))
for i, cls in enumerate(CLASS_NAMES):
    sample = df[df.ethnicity == cls].sample(5, random_state=42)["path"].tolist()
    for j, p in enumerate(sample):
        axes[i, j].imshow(np.asarray(Image.open(p).convert("RGB")))
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_ylabel(ES_POR_CLASE[cls], fontsize=11, fontweight="bold")
plt.suptitle("UTKFace: 5 ejemplos por etnia (pose/luz/edad variadas)", fontsize=13)
plt.tight_layout(); plt.show()

# Ejemplo de decodificacion del nombre
print("Ejemplo:", Path(df.iloc[0]['path']).name, "->", df.iloc[0][["age","gender","ethnicity"]].to_dict())

## 2. Preprocesamiento y división train/val/test

- **Decode JPEG → resize 128×128 → `/255` → `[0,1]`** (`float32`).
- **Split estratificado 70/15/15** por `race` (seed 42) para preservar el desbalance en los 3 sets.
- `tf.data` con `shuffle + batch(32) + prefetch`.

In [ ]:
# --- Split + tf.data ---
from sklearn.model_selection import train_test_split
import tensorflow as tf

IMG_SIZE, BATCH, SEED = 128, 32, 42

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3, try_recover_truncated=True)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

paths = df["path"].tolist(); races = df["race"].to_numpy()
p_tr, p_tmp, y_tr, y_tmp = train_test_split(paths, races, test_size=0.30, random_state=SEED, stratify=races)
p_va, p_te, y_va, y_te = train_test_split(p_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print(f"Conjuntos: entrenamiento={len(p_tr)} validación={len(p_va)} prueba={len(p_te)}")

aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.12),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
], name="augmentation")

def make_ds(paths, labels, training, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.shuffle(min(len(paths), 4096), seed=SEED)
    if augment and training:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(p_tr, y_tr, True, augment=True)
val_ds   = make_ds(p_va, y_va, False)
test_ds  = make_ds(p_te, y_te, False)

# Verificacion de normalizacion
for x, y in train_ds.take(1):
    print("batch:", x.shape, "rango:", float(x.numpy().min()), "-", float(x.numpy().max()), "labels:", y.numpy()[:8])

plt.figure(figsize=(10, 3))
for x, y in train_ds.take(1):
    for i in range(5):
        plt.subplot(1, 5, i+1); plt.imshow(x[i].numpy()); plt.title(CLASES_ES[int(y[i])], fontsize=8); plt.axis("off")
plt.suptitle("Batch de entrenamiento (128x128 normalizado [0,1])"); plt.show()

## 3. Aumento de datos (solo en entrenamiento)

Flip horizontal + rotación ±8% + zoom ±12% + contraste ±12% + traslación ±8%. Evita memorizar y simula variaciones de pose/luz.

In [ ]:
# --- Visualizar el efecto del aumento ---
sample_paths = p_tr[:2]
imgs = []
for p in sample_paths:
    raw = tf.io.read_file(p); im = tf.image.decode_jpeg(raw, channels=3); im = tf.image.resize(im, [IMG_SIZE, IMG_SIZE])
    imgs.append(tf.cast(im, tf.float32) / 255.0)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for r in range(2):
    for c in range(6):
        axes[r, c].imshow(tf.clip_by_value(aug(imgs[r], training=True), 0, 1).numpy())
        axes[r, c].axis("off")
        axes[r, c].set_title("orig." if c == 0 else f"aum. {c}", fontsize=8)
plt.suptitle("Aumento de datos: misma foto, 6 variantes"); plt.tight_layout(); plt.show()

## 4. Construcción de la CNN

**Modelo principal (transfer learning):** MobileNetV2-ImageNet congelado + `Dropout + Dense(256, relu) + Dropout + Dense(5, softmax)`.
Entrada 128×128 (reescalada a [-1,1] dentro de la red). Optimizador Adam `1e-4`, pérdida `sparse_categorical_crossentropy`.

También se define una CNN simple desde cero (4 bloques Conv+BN+Pool) como baseline didáctico.

In [ ]:
# --- Arquitecturas ---
from tensorflow import keras
from tensorflow.keras import layers

def build_simple(input_shape=(128,128,3), n=5, lr=1e-3):
    i = keras.Input(shape=input_shape)
    x = i
    for f in [32, 64, 128, 256]:
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(0.4)(x)
    o = layers.Dense(n, activation="softmax")(x)
    m = keras.Model(i, o, name="simple_cnn")
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def build_transfer(input_shape=(128,128,3), n=5, lr=1e-4, freeze=True):
    base = keras.applications.MobileNetV2(include_top=False, input_shape=input_shape, weights="imagenet", pooling="avg")
    base.trainable = not freeze
    i = keras.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(i)  # [0,1] -> [-1,1]
    x = base(x, training=not freeze)
    x = layers.Dropout(0.3)(x); x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(0.3)(x)
    o = layers.Dense(n, activation="softmax")(x)
    m = keras.Model(i, o, name="mobilenetv2_ethnicity")
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

model = build_transfer()
model.summary()

## 5. Entrenamiento

- `class_weight='balanced'` por el desbalance (White mayoritaria).
- Callbacks: `ModelCheckpoint(val_accuracy)`, `EarlyStopping(p=7)`, `ReduceLROnPlateau`, `CSVLogger`.
- 25 épocas (ajustable; en CPU usa menos épocas o el modo demo).

In [ ]:
# --- Entrenamiento ---
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

cw = compute_class_weight("balanced", classes=np.arange(5), y=np.array(y_tr))
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("Pesos por clase:", class_weight)

cbs = [
    keras.callbacks.ModelCheckpoint("ethnicity_cnn_best.h5", save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=7, restore_best_weights=True, mode="max"),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=class_weight, callbacks=cbs)


## 6. Gráficas del entrenamiento (accuracy + loss)

In [ ]:
# --- Curvas ---
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="entrenamiento"); ax[0].plot(h["val_accuracy"], label="validación")
ax[0].set_title("Exactitud"); ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].set_xlabel("Época")
ax[1].plot(h["loss"], label="entrenamiento"); ax[1].plot(h["val_loss"], label="validación")
ax[1].set_title("Pérdida"); ax[1].legend(); ax[1].grid(alpha=0.3); ax[1].set_xlabel("Época")
plt.tight_layout(); plt.savefig("training_history.png", dpi=150); plt.show()

## 7. Evaluación: métricas + matriz de confusión

Exactitud en prueba + `classification_report` (precision/recall/F1 por clase) + matriz cruda y normalizada.

In [ ]:
# --- Evaluacion ---
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

loss, acc = model.evaluate(test_ds)
print(f"Prueba: pérdida={loss:.4f} exactitud={acc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds).argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=CLASES_ES, digits=4))

for normalize, fname in [(False, "confusion_matrix.png"), (True, "confusion_matrix_norm.png")]:
    cm = confusion_matrix(y_true, y_pred)
    if normalize: cm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=CLASES_ES, yticklabels=CLASES_ES)
    plt.xlabel("Predicha"); plt.ylabel("Real")
    plt.title("Matriz de confusion" + (" (normalizada)" if normalize else ""))
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

## 8. Mínimo 10 predicciones con imágenes (12 ejemplos del test)

In [ ]:
# --- 12 predicciones ---
N = 12
imgs, labels = [], []
for x, y in test_ds.unbatch().take(N):
    imgs.append(x.numpy()); labels.append(int(y.numpy()))
imgs = np.stack(imgs)
probs = model.predict(imgs); preds = probs.argmax(axis=1)

fig, axes = plt.subplots(3, 4, figsize=(12, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(imgs[i], 0, 1))
    ok = preds[i] == labels[i]
    ax.set_title(f"Real: {CLASES_ES[labels[i]]}\nPredicha: {CLASES_ES[preds[i]]} ({probs[i][preds[i]]:.2f})",
                 fontsize=9, color="green" if ok else "red")
    ax.axis("off")
plt.suptitle("Predicciones en test (verde=acierto)", fontsize=13)
plt.tight_layout(); plt.savefig("sample_predictions.png", dpi=150); plt.show()
print(f"Aciertos: {(preds == np.array(labels)).sum()}/{N}")

## 9. Guardado del modelo entrenado

Se guarda `ethnicity_cnn.h5` + `class_names.json` para la app Streamlit (`app/app.py` carga ambos y muestra clase + probabilidad).

In [ ]:
# --- Guardado ---
import json
model.save("ethnicity_cnn.h5")
Path("class_names.json").write_text(json.dumps(CLASS_NAMES, ensure_ascii=False, indent=2))
print("Guardado: ethnicity_cnn.h5 + class_names.json")

# Para descargar desde Colab al PC (y colocarlo en model/ para Streamlit):
try:
    from google.colab import files
    files.download("ethnicity_cnn.h5"); files.download("class_names.json")
except ImportError:
    print("No es Colab: archivos en el directorio actual.")

# Inferencia de ejemplo con una imagen nueva:
from PIL import Image
def predict_path(p):
    im = Image.open(p).convert("RGB").resize((128, 128))
    x = np.expand_dims(np.asarray(im).astype("float32") / 255.0, 0)
    pr = model.predict(x, verbose=0)[0]
    return CLASES_ES[int(pr.argmax())], float(pr.max()), pr

print(predict_path(p_te[0]))  # (etiqueta, confianza, vector)


## 10. Conclusiones y limitaciones

- La CNN con transfer learning alcanza ~0.78–0.86 de accuracy en UTKFace-128px; el desbalance (White mayoritaria) baja el F1 de clases minoritarias → `class_weight` + augmentation lo mitigan pero no lo eliminan.
- Errores típicos: iluminación extrema, perfil, oclusión (gafas/mascarilla), bebés (rasgos menos diferenciados) y clase `Others` (mezcla heterogénea por definición).
- **Ética:** no desplegar para vigilancia/contratación/control; informar siempre la incertidumbre (probabilidad) y el sesgo del dataset.
- **Mejoras:** 224×224, fine-tuning progresivo, FairFace, validación cruzada, Grad-CAM para interpretabilidad.
